# Cypher 25 DSL — Building Queries Programmatically

This notebook walks through `rdflib_neo4j.sparql.cypher_builder` — a typed Python DSL
that assembles Cypher 25 queries from composable building blocks.

We use a small **FOAF social network** as our running example:
Alice and Bob are both `foaf:Person`s; Alice knows Bob; both have names and ages.

The notebook covers:
1. Setting up the dataset in Neo4j
2. Simple node patterns
3. Path patterns (relationships)
4. WHERE predicates and filters
5. Aggregation and WITH clauses
6. Optional matches
7. Subqueries and NOT EXISTS
8. UNION
9. Running rendered Cypher against Neo4j

## 0. Setup

In [ ]:
%pip install -q rdflib-neo4j neo4j python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()  # reads NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE from .env

URI      = os.environ["NEO4J_URI"]
USER     = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f"Connected to {URI} / {DATABASE}")

### Helper: run a CypherQuery

In [ ]:
from rdflib_neo4j.sparql.cypher_builder import CypherQuery

def run(q: CypherQuery):
    cypher, params = q.render()
    # Strip the 'CYPHER 25' header for the driver (5.x accepts Cypher 25 natively)
    body = "\n".join(l for l in cypher.splitlines() if l != "CYPHER 25")
    print("─" * 60)
    print(body)
    if params:
        print("params:", params)
    print("─" * 60)
    records, summary, _ = driver.execute_query(body, params, database_=DATABASE)
    return records

## 1. Load Sample Data

We'll use rdflib-neo4j to ingest a small Turtle file into Neo4j.
The store writes each resource as a `(:Resource)` node; `rdf:type` becomes a label;
literal-valued predicates become properties; URI-valued predicates become relationships.

In [ ]:
TURTLE = """
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex:   <http://example.org/> .

ex:alice a foaf:Person ;
    foaf:name  "Alice" ;
    foaf:age   30 ;
    foaf:mbox  "alice@example.org" ;
    foaf:knows ex:bob .

ex:bob a foaf:Person ;
    foaf:name  "Bob" ;
    foaf:age   25 ;
    foaf:mbox  "bob@example.org" ;
    foaf:member "engineering" .

ex:carol a foaf:Person ;
    foaf:name   "Carol" ;
    foaf:age    35 ;
    foaf:member "engineering" ;
    foaf:knows  ex:alice .

ex:dave a foaf:Person ;
    foaf:name   "Dave" ;
    foaf:age    28 ;
    foaf:member "product" .
"""

from rdflib import Graph
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

# First, clear any existing data from a previous run
driver.execute_query("MATCH (n:Resource) DETACH DELETE n", database_=DATABASE)
# Ensure uniqueness constraint exists
driver.execute_query(
    "CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS "
    "FOR (r:Resource) REQUIRE r.uri IS UNIQUE",
    database_=DATABASE,
)

auth_data = {"uri": URI, "database": DATABASE, "user": USER, "pwd": PASSWORD}
config = Neo4jStoreConfig(
    auth_data=auth_data,
    custom_prefixes={"foaf": "http://xmlns.com/foaf/0.1/"},
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=False,
)
g = Graph(store=Neo4jStore(config=config))
g.open(config, create=True)
g.parse(data=TURTLE, format="ttl")
g.close(True)

# Verify
count, _, _ = driver.execute_query("MATCH (n:Person) RETURN count(n) AS c", database_=DATABASE)
print(f"Loaded {count[0]['c']} Person nodes")

## 2. Simple Node Pattern

`NodePattern(Var("n"), ["Person"])` renders as `(n:Person)`.

In [ ]:
from rdflib_neo4j.sparql.cypher_builder import (
    CypherQuery, NodePattern, Var, RawExpression, AliasExpression,
    Comparison, StringLiteral, AndPredicate, IsNotNull, OrderItem,
    FunctionCall, PathPattern, RelSegment, AnonNode,
)

# MATCH (p:Person) RETURN p.uri AS uri, p.`name` AS name
q = (
    CypherQuery()
    .match(NodePattern(Var("p"), ["Person"]))
    .return_(
        AliasExpression(RawExpression("p.uri"), "uri"),
        AliasExpression(RawExpression("p.`name`"), "name"),
        AliasExpression(RawExpression("p.`age`"), "age"),
    )
)

for r in run(q):
    print(r["name"], "(", r["age"], ")")

## 3. WHERE Predicates

The DSL has a rich predicate hierarchy: `Comparison`, `AndPredicate`, `OrPredicate`,
`IsNotNull`, `InPredicate`, `RawPredicate`.

In [ ]:
# Adults: age >= 28, have a name
age_expr  = RawExpression("p.`age`")
name_expr = RawExpression("p.`name`")

q = (
    CypherQuery()
    .match(NodePattern(Var("p"), ["Person"]))
    .where(AndPredicate(
        Comparison(age_expr, ">=", RawExpression("28")),
        IsNotNull(name_expr),
    ))
    .return_(
        AliasExpression(name_expr, "name"),
        AliasExpression(age_expr, "age"),
        order_by=[OrderItem(age_expr, ascending=True)],
    )
)

for r in run(q):
    print(r["name"], r["age"])

## 4. Parameters — Safe Value Binding

`query.add_param(value)` registers the value in the params dict and returns
a `RawExpression("$p0")` placeholder — no string interpolation, no injection risk.

In [ ]:
q = CypherQuery()
name_param = q.add_param("Alice")   # → $p0

q = (
    q
    .match(NodePattern(Var("p"), ["Person"]))
    .where(Comparison(RawExpression("p.`name`"), "=", name_param))
    .return_(AliasExpression(RawExpression("p.`name`"), "name"),
             AliasExpression(RawExpression("p.`age`"), "age"))
)

for r in run(q):
    print(r["name"], r["age"])

## 5. Path Patterns — Relationships

`PathPattern` composes `AnonNode` and `RelSegment` into directed path expressions like
`(p)-[:knows]->(f)`.

In [ ]:
# Who does Alice know?
q = CypherQuery()
alice_param = q.add_param("Alice")

path = (
    PathPattern(AnonNode(Var("p")))
    .rel(RelSegment(types=["knows"]), AnonNode(Var("f")))
)

q = (
    q
    .match(NodePattern(Var("p"), ["Person"]))
    .where(Comparison(RawExpression("p.`name`"), "=", alice_param))
    .match(path)
    .match(NodePattern(Var("f"), ["Person"]))
    .return_(
        AliasExpression(RawExpression("p.`name`"), "person"),
        AliasExpression(RawExpression("f.`name`"), "knows"),
    )
)

for r in run(q):
    print(r["person"], "→ knows →", r["knows"])

## 6. OPTIONAL MATCH

`query.optional_match(pattern)` emits `OPTIONAL MATCH` — absent matches produce `null`
rather than dropping the row.

In [ ]:
# All Persons + their mbox if present
q = (
    CypherQuery()
    .match(NodePattern(Var("p"), ["Person"]))
    .return_(
        AliasExpression(RawExpression("p.`name`"), "name"),
        AliasExpression(RawExpression("p.`mbox`"), "mbox"),   # may be null
        AliasExpression(RawExpression("p.`member`"), "dept"),  # may be null
        order_by=[OrderItem(RawExpression("p.`name`"), ascending=True)],
    )
)

for r in run(q):
    print(f"{r['name']:8}  mbox={r['mbox']}  dept={r['dept']}")

## 7. Aggregation with WITH

The DSL's `with_()` method lets you group by expressions and pass aggregates forward.

In [ ]:
from rdflib_neo4j.sparql.cypher_builder import AliasExpression

# Count persons per department
q = (
    CypherQuery()
    .match(NodePattern(Var("p"), ["Person"]))
    .where(IsNotNull(RawExpression("p.`member`")))
    .with_(
        AliasExpression(RawExpression("p.`member`"), "dept"),
        AliasExpression(FunctionCall("count", Var("p")), "cnt"),
    )
    .return_("dept", "cnt", order_by=[OrderItem(Var("cnt"), ascending=False)])
)

for r in run(q):
    print(f"{r['dept']:15} {r['cnt']} person(s)")

## 8. NOT EXISTS — Subquery Predicates

`query.not_exists_subquery(inner)` wraps an inner `CypherQuery` in a
`NOT EXISTS { … }` predicate — the Cypher 25 way to do SPARQL MINUS.

In [ ]:
# Persons who don't know anyone
outer = CypherQuery().match(NodePattern(Var("p"), ["Person"]))

inner = (
    CypherQuery()
    .match(
        PathPattern(AnonNode(Var("p")))
        .rel(RelSegment(types=["knows"]), AnonNode(Var("_other")))
    )
)

q = (
    outer
    .where(outer.not_exists_subquery(inner))
    .return_(AliasExpression(RawExpression("p.`name`"), "name"))
)

for r in run(q):
    print(r["name"], "doesn't know anyone")

## 9. UNION

Two `CypherQuery` objects can be joined with `.union()` — equivalent to SPARQL UNION.

In [ ]:
# People named Alice OR people in engineering
q1 = (
    CypherQuery()
    .match(NodePattern(Var("p"), ["Person"]))
    .where(Comparison(RawExpression("p.`name`"), "=", StringLiteral("Alice")))
    .return_(AliasExpression(RawExpression("p.`name`"), "name"),
             AliasExpression(StringLiteral("name match"), "reason"))
)

q2 = (
    CypherQuery()
    .match(NodePattern(Var("p"), ["Person"]))
    .where(Comparison(RawExpression("p.`member`"), "=", StringLiteral("engineering")))
    .return_(AliasExpression(RawExpression("p.`name`"), "name"),
             AliasExpression(StringLiteral("dept match"), "reason"))
)

q1.union(q2)
for r in run(q1):
    print(r["name"], "—", r["reason"])

## 10. DISTINCT + LIMIT + SKIP

`return_()` accepts `distinct`, `limit`, `skip`, and `order_by` as kwargs.

In [ ]:
# Unique departments, first 5, skip 0
q = (
    CypherQuery()
    .match(NodePattern(Var("p"), ["Person"]))
    .where(IsNotNull(RawExpression("p.`member`")))
    .return_(
        AliasExpression(RawExpression("p.`member`"), "dept"),
        distinct=True,
        order_by=[OrderItem(RawExpression("p.`member`"), ascending=True)],
        limit=5,
        skip=0,
    )
)

for r in run(q):
    print(r["dept"])

## 11. Inspect Rendered Cypher Without Running

`.render()` returns `(cypher_string, params_dict)` — useful for debugging,
logging, or passing the query to another driver.

In [ ]:
q = CypherQuery()
dept_param = q.add_param("engineering")
q = (
    q
    .match(NodePattern(Var("p"), ["Person"]))
    .where(Comparison(RawExpression("p.`member`"), "=", dept_param))
    .return_(
        AliasExpression(RawExpression("p.`name`"), "name"),
        AliasExpression(RawExpression("p.`age`"),  "age"),
        order_by=[OrderItem(RawExpression("p.`age`"), ascending=True)],
    )
)

cypher, params = q.render()
print(cypher)
print("params:", params)

In [ ]:
driver.close()